In [ ]:
print("Hello")

In [ ]:
import pandas as pd
data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")

In [ ]:
data

In [ ]:
pd.unique(data['equip_type_famille'])

In [ ]:
data['equip_type_famille'].str.lower().str.contains("site d'activités aquatiques et nautiques").sum()

In [ ]:

data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]


colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()



In [ ]:
data_sud = data_nautique[data_nautique['dep_code'].str.lower().isin(["13", "6"])]
data_sud['aps_name'].str.lower().str.contains("surf|ski|voile|foil|wake").sum()

In [ ]:
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]

data_nautique

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Séparer lat/lon (si equip_coordonnees est du type "lat, lon")
data_nautique[['lat', 'lon']] = data_nautique['equip_coordonnees'].str.split(',', expand=True).astype(float)

data_nautique['lat'] = data_nautique['lat'].dropna()
data_nautique['lon'] = data_nautique['lon'].dropna()
data_nautique['equip_nom'] = data_nautique['equip_nom'].dropna()
# Créer le GeoDataFrame
geometry = [Point(lon, lat) for lat, lon in zip(data_nautique['lat'], data_nautique['lon'])]
gdf = gpd.GeoDataFrame(data_nautique, geometry=geometry, crs=2154)

In [ ]:


import folium
m = folium.Map(location=[46.6, 2.3], zoom_start=6)

for _, row in data_nautique.dropna(subset=['lat', 'lon']).iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['aps_name']
    ).add_to(m)

m
